# 46장 · 검색 규칙 변경과 실패 단계 진단

검색 규칙을 바꾼 뒤에는 문서가 다시 검색되는지와 그 문서로 옳은 답을 만드는지를 따로 검사합니다.

기준선에서 실패한 Q03을 대상으로 ‘실패’와 ‘오류’를 검색 규칙에 추가합니다. 입력 문서, 생성 함수, k=2를 고정하고 p3 검색 여부와 인용 결과를 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요.


## 키워드 변경 전후의 문단 적중

완성 실험 기록｜변경 전 Q03은 검색 1회, 빈 목록, 생성 0회입니다. 후보에서는 같은 문장에 오류 낱말을 인식해 p3을 가져옵니다. 여기서는 제공 기록에 맞춰 검색 조건 한 가지를 재현합니다.


### 셀 01 · 키워드 규칙 변경과 적중

Q03에 어떤 낱말이 들어 있으며 그 조건에서 어느 문단을 돌려주는지 확인하세요.


In [1]:
error_doc = {'id': 'course-guide-v1:p3',
 'text': '제출 오류가 발생하면 오류 화면과 발생 시간을 기록하고 담당 강사에게 문의합니다. 비밀키가 보이는 화면은 공유하지 '
         '않습니다.',
 'source': '가상 과정 운영 안내 · 45일 수업용 페이지판 (source-pages.html)',
 'page': 1}
question = '과제 업로드가 실패했어요. 어떻게 해야 하나요?'
found = [error_doc] if any(word in question for word in ('오류','실패')) else []
print([d['id'] for d in found])

['course-guide-v1:p3']


course-guide-v1:p3을 찾았습니다. 원문에는 오류 화면과 발생 시간 기록, 강사 문의, 비밀키 공유 금지가 담겨 있습니다.


## 검색 성공 뒤의 인용 실패

고정 생성기는 Q01에서 썼던 p2를 인용합니다. 검색은 p3을 돌려주는데 p2 인용을 사용자에게 보여 주면 출처가 어긋납니다. 그래서 이번에는 기준선을 유지하고 생성 수정을 별도의 실험으로 넘깁니다.


### 셀 02 · 검색 이후 실패 단계 비교

두 행은 같은 Q03의 제공 로컬 실행 기록입니다. 검색 정책 외에 무엇을 고정했는지 기억하며 살펴보세요.

Q03에서 검색이 허용한 문단은 p3인데 고정 답변의 대괄호 안에는 p2가 있습니다. ‘검색 성공’에는 p3을, ‘인용 검사 실패’에는 p2를 적으세요. 검색 문서를 찾았다는 이유로 이 답을 사용자에게 내보내지 않습니다.


In [2]:
before = {'retrieved_ids': [],
 'result': {'status': 'no_evidence',
            'answer': '제공된 자료에서 확인할 수 없습니다.',
            'sources': [],
            'generation_calls': 0,
            'retrieved_ids': [],
            'seconds': 1e-06}}
after = {'retrieved_ids': ['course-guide-v1:p3'],
 'result': {'status': 'invalid_citation',
            'answer': '출처 표시를 확인해야 합니다.',
            'sources': [],
            'generation_calls': 1,
            'retrieved_ids': ['course-guide-v1:p3'],
            'seconds': 4e-06}}
print(before['retrieved_ids'], before['result']['status'])
print(after['retrieved_ids'], after['result']['status'])

[] no_evidence
['course-guide-v1:p3'] invalid_citation


빈 목록의 no_evidence가 p3과 invalid_citation으로 바뀌었습니다. 검색 성공과 사용자가 읽을 수 있는 답변 성공은 따로 기록해야 합니다. 자기 팀도 가장 자주 틀리는 단계 하나부터 비교하세요.

수정 전의 no_evidence와 수정 뒤의 invalid_citation은 서로 다른 결함입니다. 두 번째 실행은 p3을 찾는 데 성공했습니다. 이 상태에서 검색어 규칙을 다시 넓히기보다 어떤 인용 ID를 답변에 넣었는지 확인하면 다음 수정 범위를 좁힐 수 있습니다.


### 다음 실패가 드러나는 시점

변경 전에는 근거가 없어 생성을 0회로 마쳤습니다. 변경 후에는 p3을 찾아 생성까지 1회 실행했지만 인용 검사에서 멈췄습니다. 두 상태를 나란히 읽으면 다음에 고칠 대상이 검색 키워드에서 인용으로 바뀐 것을 알 수 있습니다.


## 돌아보기

이 실험의 선택 후보는 error_keyword, 실제 유지한 방식은 baseline입니다. 다음 장을 맡은 팀원이 같은 조건에서 시작할 수 있도록 유지한 방식과 그 이유를 함께 건네세요.
